# 310 — Cell-Line Transcriptomic Program Discovery

Identify transcriptomic programs from gene expression in the harmonized
712-model cell-line cohort. Notebook 308 supplies the expression profiles and
`selected_phenotype`, the median `LN_IC50` across compounds tested for each model.

## Analysis

- Select the 5,000 most variable genes and fit ICA as the primary discovery
  method. Use multiple initializations to assess component stability.
- Fit NMF on the same expression data and compare its components with ICA
  using model scores and gene loadings.
- After defining the programs, test recoverable ICA components for association
  with `selected_phenotype` using Spearman correlation and Benjamini–Hochberg
  correction (`q < 0.05`).

## Outputs

- `310_ica_program_scores.parquet`
- `310_ica_program_loadings.parquet`
- `310_program_phenotype_associations.csv`
- `310_program_discovery_metadata.json`

Notebook 311 evaluates the robustness of the resulting candidate associations.

In [1]:
# =============================================================================
# Imports
# =============================================================================

from pancancer_epigenetics.utils.paths import (Paths, project_relative_path)

import json

import numpy as np
import pandas as pd

from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr

from sklearn.decomposition import FastICA, NMF

In [2]:
# =============================================================================
# Input and output paths
# =============================================================================

TRANSCRIPTOME_PHENOTYPE_PATH = (
    Paths.pharmacology
    / "308_transcriptome_phenotype_dataset.parquet"
)

OUTPUT_DIR = Paths.cellline_programs

In [3]:
# =============================================================================
# Load frozen transcriptome–phenotype dataset
# =============================================================================

transcriptome_phenotype = pd.read_parquet(
    TRANSCRIPTOME_PHENOTYPE_PATH
)

print("Transcriptome–phenotype dataset:", transcriptome_phenotype.shape)

Transcriptome–phenotype dataset: (712, 19195)


In [4]:
# =============================================================================
# Separate transcriptome and phenotype
# =============================================================================

model_ids = transcriptome_phenotype["ModelID"].copy()

expression = transcriptome_phenotype.drop(
    columns=[
        "ModelID",
        "selected_phenotype",
    ]
)

selected_phenotype = transcriptome_phenotype[
    "selected_phenotype"
].copy()

print("Expression matrix:", expression.shape)

Expression matrix: (712, 19193)


In [5]:
# =============================================================================
# Remove zero-variance genes
# =============================================================================

expression = expression.loc[
    :,
    expression.var(axis=0).gt(0),
]

print("Expression matrix after zero-variance removal:", expression.shape)

Expression matrix after zero-variance removal: (712, 19183)


In [6]:
# =============================================================================
# Program-discovery parameters
# =============================================================================

N_DISCOVERY_GENES = 5000
N_COMPONENTS = 50

N_ICA_RUNS = 20
RANDOM_SEED = 20260812

In [7]:
# =============================================================================
# Select discovery gene universe
# =============================================================================

gene_variance = expression.var(axis=0)

discovery_genes = (
    gene_variance
    .nlargest(N_DISCOVERY_GENES)
    .index
)

discovery_expression = expression.loc[
    :,
    discovery_genes,
].copy()

print("Discovery expression matrix:", discovery_expression.shape)

Discovery expression matrix: (712, 5000)


In [8]:
# =============================================================================
# Prepare ICA discovery matrix
# =============================================================================

ica_expression = discovery_expression.to_numpy(
    dtype=np.float64,
    copy=True,
)

ica_seeds = (
    RANDOM_SEED
    + np.arange(N_ICA_RUNS)
)

In [9]:
# =============================================================================
# Run multistart ICA
# =============================================================================

ica_runs = []

for seed in ica_seeds:
    model = FastICA(
        n_components=N_COMPONENTS,
        whiten="unit-variance",
        random_state=int(seed),
        max_iter=2000,
    )

    scores = model.fit_transform(ica_expression)

    ica_runs.append(
        {
            "seed": int(seed),
            "scores": scores,
            "loadings": model.mixing_.copy(),
            "n_iter": model.n_iter_,
        }
    )

print("ICA runs completed:", len(ica_runs))
print(
    "Maximum iterations used:",
    max(run["n_iter"] for run in ica_runs),
)

ICA runs completed: 20
Maximum iterations used: 101


In [10]:
# =============================================================================
# Define ICA component matching
# =============================================================================

def match_ica_components(reference_loadings, target_loadings):
    reference_centered = (
        reference_loadings
        - reference_loadings.mean(axis=0, keepdims=True)
    )
    target_centered = (
        target_loadings
        - target_loadings.mean(axis=0, keepdims=True)
    )

    reference_unit = (
        reference_centered
        / np.linalg.norm(
            reference_centered,
            axis=0,
            keepdims=True,
        )
    )
    target_unit = (
        target_centered
        / np.linalg.norm(
            target_centered,
            axis=0,
            keepdims=True,
        )
    )

    loading_correlations = (
        reference_unit.T
        @ target_unit
    )

    reference_indices, target_indices = (
        linear_sum_assignment(
            -np.abs(loading_correlations)
        )
    )

    matched_correlations = loading_correlations[
        reference_indices,
        target_indices,
    ]

    return (
        target_indices,
        matched_correlations,
    )

In [11]:
# =============================================================================
# Select central ICA reference run
# =============================================================================

ica_run_stability = []

for reference_index, reference_run in enumerate(ica_runs):
    pairwise_stability = []

    for target_index, target_run in enumerate(ica_runs):
        if reference_index == target_index:
            continue

        _, matched_correlations = match_ica_components(
            reference_run["loadings"],
            target_run["loadings"],
        )

        pairwise_stability.append(
            np.median(np.abs(matched_correlations))
        )

    ica_run_stability.append(
        {
            "run_index": reference_index,
            "seed": reference_run["seed"],
            "median_pairwise_stability": np.median(
                pairwise_stability
            ),
        }
    )

ica_run_stability = pd.DataFrame(
    ica_run_stability
).sort_values(
    "median_pairwise_stability",
    ascending=False,
)

reference_run_index = int(
    ica_run_stability.iloc[0]["run_index"]
)

reference_ica_run = ica_runs[
    reference_run_index
]

ica_run_stability.head()

,run_index,seed,median_pairwise_stability
8,8,20260820,0.994899
16,16,20260828,0.994819
6,6,20260818,0.994733
2,2,20260814,0.994057
12,12,20260824,0.994025


In [12]:
# =============================================================================
# Quantify component-level ICA stability
# =============================================================================

component_stability = []

for target_index, target_run in enumerate(ica_runs):
    if target_index == reference_run_index:
        continue

    matched_indices, matched_correlations = match_ica_components(
        reference_ica_run["loadings"],
        target_run["loadings"],
    )

    for component_index, (
        matched_index,
        loading_correlation,
    ) in enumerate(
        zip(
            matched_indices,
            matched_correlations,
        )
    ):
        sign = np.sign(loading_correlation)

        score_correlation = np.corrcoef(
            reference_ica_run["scores"][:, component_index],
            target_run["scores"][:, matched_index] * sign,
        )[0, 1]

        component_stability.append(
            {
                "component_index": component_index,
                "loading_correlation": abs(loading_correlation),
                "score_correlation": score_correlation,
            }
        )

component_stability = pd.DataFrame(component_stability)

In [13]:
# =============================================================================
# Summarize component-level ICA stability
# =============================================================================

ica_component_stability = (
    component_stability
    .groupby("component_index")
    .agg(
        median_loading_correlation=(
            "loading_correlation",
            "median",
        ),
        minimum_loading_correlation=(
            "loading_correlation",
            "min",
        ),
        median_score_correlation=(
            "score_correlation",
            "median",
        ),
        minimum_score_correlation=(
            "score_correlation",
            "min",
        ),
    )
    .reset_index()
)

ica_component_stability.sort_values(
    [
        "median_loading_correlation",
        "median_score_correlation",
    ]
).head(10)

,component_index,median_loading_correlation,minimum_loading_correlation,median_score_correlation,minimum_score_correlation
2,2,0.545893,0.181767,0.585860,0.386493
4,4,0.813910,0.241862,0.883447,-0.023561
17,17,0.828310,0.514682,0.814200,0.342305
24,24,0.829392,0.396230,0.860380,0.363672
27,27,0.847782,0.393987,0.862467,0.424554
45,45,0.855122,0.127393,0.873847,0.273687
21,21,0.864978,0.635644,0.904904,0.621253
36,36,0.886663,0.394542,0.861634,0.303799
12,12,0.891626,0.591740,0.906883,0.584942
31,31,0.892929,0.244917,0.924645,0.341655


In [14]:
# =============================================================================
# Inspect ICA stability distribution
# =============================================================================

ica_stability_distribution = (
    ica_component_stability[
        [
            "median_loading_correlation",
            "median_score_correlation",
        ]
    ]
    .describe()
    .loc[
        [
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

ica_stability_distribution

,median_loading_correlation,median_score_correlation
min,0.545893,0.585860
25%,0.917856,0.912007
50%,0.995379,0.990678
75%,0.999457,0.998191
max,0.999893,0.999651


In [15]:
# =============================================================================
# Define ICA component recoverability
# =============================================================================

ICA_STABILITY_THRESHOLD = 0.70

ica_component_stability["is_recoverable"] = (
    ica_component_stability[
        "median_loading_correlation"
    ].ge(ICA_STABILITY_THRESHOLD)
    & ica_component_stability[
        "median_score_correlation"
    ].ge(ICA_STABILITY_THRESHOLD)
)

ica_component_stability[
    "is_recoverable"
].value_counts()

is_recoverable
True     49
False     1
Name: count, dtype: int64

In [16]:
# =============================================================================
# Orient ICA components deterministically
# =============================================================================

ica_scores = reference_ica_run["scores"].copy()
ica_loadings = reference_ica_run["loadings"].copy()

for component_index in range(N_COMPONENTS):
    anchor_gene_index = np.argmax(
        np.abs(ica_loadings[:, component_index])
    )

    orientation = np.sign(
        ica_loadings[
            anchor_gene_index,
            component_index,
        ]
    )

    ica_scores[:, component_index] *= orientation
    ica_loadings[:, component_index] *= orientation

In [17]:
# =============================================================================
# Construct ICA program matrices
# =============================================================================

ica_program_ids = [
    f"ICA_PROGRAM_{index + 1:02d}"
    for index in range(N_COMPONENTS)
]

ica_program_scores = pd.DataFrame(
    ica_scores,
    index=model_ids,
    columns=ica_program_ids,
).reset_index(
    names="ModelID"
)

ica_program_loadings = pd.DataFrame(
    ica_loadings,
    index=discovery_expression.columns,
    columns=ica_program_ids,
).reset_index(
    names="gene"
)

print("ICA program scores  :", ica_program_scores.shape)
print("ICA program loadings:", ica_program_loadings.shape)

ICA program scores  : (712, 51)
ICA program loadings: (5000, 51)


In [18]:
# =============================================================================
# Construct ICA program metadata
# =============================================================================

ica_program_metadata = (
    ica_component_stability
    .assign(
        program_id=[
            f"ICA_PROGRAM_{index + 1:02d}"
            for index in ica_component_stability[
                "component_index"
            ]
        ]
    )
    [
        [
            "program_id",
            "component_index",
            "median_loading_correlation",
            "minimum_loading_correlation",
            "median_score_correlation",
            "minimum_score_correlation",
            "is_recoverable",
        ]
    ]
)

ica_program_metadata

,program_id,component_index,median_loading_correlation,minimum_loading_correlation,median_score_correlation,minimum_score_correlation,is_recoverable
0,ICA_PROGRAM_01,0,0.996324,0.987724,0.995478,0.985053,True
1,ICA_PROGRAM_02,1,0.916594,0.568015,0.948621,0.597908,True
2,ICA_PROGRAM_03,2,0.545893,0.181767,0.585860,0.386493,False
3,ICA_PROGRAM_04,3,0.922845,0.626382,0.909887,0.658965,True
4,ICA_PROGRAM_05,4,0.813910,0.241862,0.883447,-0.023561,True
5,ICA_PROGRAM_06,5,0.988751,0.884062,0.984288,0.775874,True
6,ICA_PROGRAM_07,6,0.998106,0.993540,0.995430,0.983490,True
7,ICA_PROGRAM_08,7,0.999678,0.999368,0.999536,0.999327,True
8,ICA_PROGRAM_09,8,0.999893,0.999742,0.999651,0.999129,True
9,ICA_PROGRAM_10,9,0.999857,0.999724,0.999510,0.998789,True


In [19]:
# =============================================================================
# Prepare NMF discovery matrix
# =============================================================================

N_NMF_RUNS = 20

nmf_expression = discovery_expression.to_numpy(
    dtype=np.float64,
    copy=True,
)

nmf_seeds = (
    RANDOM_SEED
    + np.arange(N_NMF_RUNS)
)

In [20]:
# =============================================================================
# Run multistart NMF
# =============================================================================

nmf_runs = []

for seed in nmf_seeds:
    model = NMF(
        n_components=N_COMPONENTS,
        init="nndsvdar",
        random_state=int(seed),
        max_iter=2000,
    )

    scores = model.fit_transform(nmf_expression)

    nmf_runs.append(
        {
            "seed": int(seed),
            "scores": scores,
            "loadings": model.components_.T.copy(),
            "reconstruction_error": model.reconstruction_err_,
            "n_iter": model.n_iter_,
        }
    )

print("NMF runs completed:", len(nmf_runs))
print(
    "Maximum iterations used:",
    max(run["n_iter"] for run in nmf_runs),
)

NMF runs completed: 20
Maximum iterations used: 1733


In [21]:
# =============================================================================
# Summarize NMF run diagnostics
# =============================================================================

nmf_run_diagnostics = pd.DataFrame(
    {
        "seed": [run["seed"] for run in nmf_runs],
        "n_iter": [run["n_iter"] for run in nmf_runs],
        "reconstruction_error": [
            run["reconstruction_error"]
            for run in nmf_runs
        ],
    }
)

nmf_run_diagnostics.describe()

,seed,n_iter,reconstruction_error
count,2.000000e+01,20.000000,20.000000
mean,2.026082e+07,962.550000,1934.293795
std,5.916080e+00,328.198027,0.256569
min,2.026081e+07,540.000000,1933.874512
25%,2.026082e+07,693.250000,1934.113772
50%,2.026082e+07,931.500000,1934.317685
75%,2.026083e+07,1114.250000,1934.467199
max,2.026083e+07,1733.000000,1934.652369


In [22]:
# =============================================================================
# Define NMF component matching
# =============================================================================

def match_nmf_components(reference_loadings, target_loadings):
    reference_unit = (
        reference_loadings
        / np.linalg.norm(
            reference_loadings,
            axis=0,
            keepdims=True,
        )
    )

    target_unit = (
        target_loadings
        / np.linalg.norm(
            target_loadings,
            axis=0,
            keepdims=True,
        )
    )

    loading_similarity = (
        reference_unit.T
        @ target_unit
    )

    reference_indices, target_indices = (
        linear_sum_assignment(
            -loading_similarity
        )
    )

    matched_similarity = loading_similarity[
        reference_indices,
        target_indices,
    ]

    return (
        target_indices,
        matched_similarity,
    )

In [23]:
# =============================================================================
# Select central NMF reference run
# =============================================================================

nmf_run_stability = []

for reference_index, reference_run in enumerate(nmf_runs):
    pairwise_stability = []

    for target_index, target_run in enumerate(nmf_runs):
        if reference_index == target_index:
            continue

        _, matched_similarity = match_nmf_components(
            reference_run["loadings"],
            target_run["loadings"],
        )

        pairwise_stability.append(
            np.median(matched_similarity)
        )

    nmf_run_stability.append(
        {
            "run_index": reference_index,
            "seed": reference_run["seed"],
            "median_pairwise_stability": np.median(
                pairwise_stability
            ),
        }
    )

nmf_run_stability = pd.DataFrame(
    nmf_run_stability
).sort_values(
    "median_pairwise_stability",
    ascending=False,
)

reference_nmf_run_index = int(
    nmf_run_stability.iloc[0]["run_index"]
)

reference_nmf_run = nmf_runs[
    reference_nmf_run_index
]

nmf_run_stability.head()

,run_index,seed,median_pairwise_stability
12,12,20260824,0.998507
1,1,20260813,0.998422
13,13,20260825,0.998388
7,7,20260819,0.998179
8,8,20260820,0.998176


In [24]:
# =============================================================================
# Quantify component-level NMF stability
# =============================================================================

nmf_component_stability_records = []

for target_index, target_run in enumerate(nmf_runs):
    if target_index == reference_nmf_run_index:
        continue

    matched_indices, matched_similarity = match_nmf_components(
        reference_nmf_run["loadings"],
        target_run["loadings"],
    )

    for component_index, (
        matched_index,
        loading_similarity,
    ) in enumerate(
        zip(
            matched_indices,
            matched_similarity,
        )
    ):
        score_correlation = np.corrcoef(
            reference_nmf_run["scores"][:, component_index],
            target_run["scores"][:, matched_index],
        )[0, 1]

        nmf_component_stability_records.append(
            {
                "component_index": component_index,
                "loading_similarity": loading_similarity,
                "score_correlation": score_correlation,
            }
        )

nmf_component_stability_records = pd.DataFrame(
    nmf_component_stability_records
)

In [25]:
# =============================================================================
# Summarize component-level NMF stability
# =============================================================================

nmf_component_stability = (
    nmf_component_stability_records
    .groupby("component_index")
    .agg(
        median_loading_similarity=(
            "loading_similarity",
            "median",
        ),
        minimum_loading_similarity=(
            "loading_similarity",
            "min",
        ),
        median_score_correlation=(
            "score_correlation",
            "median",
        ),
        minimum_score_correlation=(
            "score_correlation",
            "min",
        ),
    )
    .reset_index()
)

nmf_component_stability.sort_values(
    [
        "median_loading_similarity",
        "median_score_correlation",
    ]
).head(10)

,component_index,median_loading_similarity,minimum_loading_similarity,median_score_correlation,minimum_score_correlation
45,45,0.740554,0.539240,0.186858,0.019083
43,43,0.797918,0.582094,0.444935,-0.123568
22,22,0.874489,0.407873,0.727989,0.021530
28,28,0.975942,0.498425,0.960473,-0.059171
47,47,0.977799,0.948519,0.930725,0.902732
2,2,0.983224,0.869916,0.950460,0.768048
0,0,0.988004,0.858935,0.971165,0.398187
31,31,0.988503,0.843310,0.947536,0.456479
26,26,0.990646,0.986680,0.963969,0.948197
9,9,0.992020,0.954745,0.975548,0.889159


In [26]:
# =============================================================================
# Inspect NMF stability distribution
# =============================================================================

nmf_stability_distribution = (
    nmf_component_stability[
        [
            "median_loading_similarity",
            "median_score_correlation",
        ]
    ]
    .describe()
    .loc[
        [
            "min",
            "25%",
            "50%",
            "75%",
            "max",
        ]
    ]
)

nmf_stability_distribution

,median_loading_similarity,median_score_correlation
min,0.740554,0.186858
25%,0.992639,0.984145
50%,0.998053,0.997369
75%,0.999733,0.999366
max,0.999946,0.999960


In [27]:
# =============================================================================
# Define NMF component recoverability
# =============================================================================

NMF_STABILITY_THRESHOLD = 0.70

nmf_component_stability["is_recoverable"] = (
    nmf_component_stability[
        "median_loading_similarity"
    ].ge(NMF_STABILITY_THRESHOLD)
    & nmf_component_stability[
        "median_score_correlation"
    ].ge(NMF_STABILITY_THRESHOLD)
)

nmf_component_stability[
    "is_recoverable"
].value_counts()

is_recoverable
True     48
False     2
Name: count, dtype: int64

In [28]:
# =============================================================================
# Construct NMF program matrices
# =============================================================================

nmf_program_ids = [
    f"NMF_PROGRAM_{index + 1:02d}"
    for index in range(N_COMPONENTS)
]

nmf_program_scores = pd.DataFrame(
    reference_nmf_run["scores"],
    index=model_ids,
    columns=nmf_program_ids,
).reset_index(
    names="ModelID"
)

nmf_program_loadings = pd.DataFrame(
    reference_nmf_run["loadings"],
    index=discovery_expression.columns,
    columns=nmf_program_ids,
).reset_index(
    names="gene"
)

print("NMF program scores  :", nmf_program_scores.shape)
print("NMF program loadings:", nmf_program_loadings.shape)

NMF program scores  : (712, 51)
NMF program loadings: (5000, 51)


In [29]:
# =============================================================================
# Construct NMF program metadata
# =============================================================================

nmf_program_metadata = (
    nmf_component_stability
    .assign(
        program_id=[
            f"NMF_PROGRAM_{index + 1:02d}"
            for index in nmf_component_stability[
                "component_index"
            ]
        ]
    )
    [
        [
            "program_id",
            "component_index",
            "median_loading_similarity",
            "minimum_loading_similarity",
            "median_score_correlation",
            "minimum_score_correlation",
            "is_recoverable",
        ]
    ]
)

nmf_program_metadata

,program_id,component_index,median_loading_similarity,minimum_loading_similarity,median_score_correlation,minimum_score_correlation,is_recoverable
0,NMF_PROGRAM_01,0,0.988004,0.858935,0.971165,0.398187,True
1,NMF_PROGRAM_02,1,0.993442,0.961939,0.988426,0.939520,True
2,NMF_PROGRAM_03,2,0.983224,0.869916,0.950460,0.768048,True
3,NMF_PROGRAM_04,3,0.999790,0.998617,0.999333,0.997544,True
4,NMF_PROGRAM_05,4,0.998433,0.982402,0.994350,0.912905,True
5,NMF_PROGRAM_06,5,0.999875,0.999269,0.999735,0.999158,True
6,NMF_PROGRAM_07,6,0.992417,0.936653,0.983336,0.840482,True
7,NMF_PROGRAM_08,7,0.999918,0.999567,0.999922,0.999823,True
8,NMF_PROGRAM_09,8,0.999820,0.997639,0.999522,0.996641,True
9,NMF_PROGRAM_10,9,0.992020,0.954745,0.975548,0.889159,True


In [30]:
# =============================================================================
# Define recoverable programs for cross-method comparison
# =============================================================================

recoverable_ica_programs = (
    ica_program_metadata
    .loc[
        ica_program_metadata["is_recoverable"],
        "program_id",
    ]
    .tolist()
)

recoverable_nmf_programs = (
    nmf_program_metadata
    .loc[
        nmf_program_metadata["is_recoverable"],
        "program_id",
    ]
    .tolist()
)

In [31]:
# =============================================================================
# Compute pairwise ICA–NMF structural concordance
# =============================================================================

cross_method_records = []

for ica_program in recoverable_ica_programs:
    ica_loading = ica_program_loadings[
        ica_program
    ].to_numpy()

    ica_positive = np.clip(
        ica_loading,
        a_min=0,
        a_max=None,
    )
    ica_negative = np.clip(
        -ica_loading,
        a_min=0,
        a_max=None,
    )

    ica_score = ica_program_scores[
        ica_program
    ].to_numpy()

    for nmf_program in recoverable_nmf_programs:
        nmf_loading = nmf_program_loadings[
            nmf_program
        ].to_numpy()

        positive_similarity = (
            ica_positive @ nmf_loading
        ) / (
            np.linalg.norm(ica_positive)
            * np.linalg.norm(nmf_loading)
        )

        negative_similarity = (
            ica_negative @ nmf_loading
        ) / (
            np.linalg.norm(ica_negative)
            * np.linalg.norm(nmf_loading)
        )

        matched_pole = (
            "positive"
            if positive_similarity >= negative_similarity
            else "negative"
        )

        loading_similarity = max(
            positive_similarity,
            negative_similarity,
        )

        score_correlation, _ = spearmanr(
            ica_score,
            nmf_program_scores[
                nmf_program
            ].to_numpy(),
        )

        aligned_score_correlation = (
            score_correlation
            if matched_pole == "positive"
            else -score_correlation
        )

        cross_method_records.append(
            {
                "ica_program": ica_program,
                "nmf_program": nmf_program,
                "ica_pole": matched_pole,
                "loading_similarity": loading_similarity,
                "score_spearman": score_correlation,
                "aligned_score_spearman": (
                    aligned_score_correlation
                ),
            }
        )

cross_method_concordance = pd.DataFrame(
    cross_method_records
)

In [32]:
# =============================================================================
# Summarize best ICA–NMF matches across structural views
# =============================================================================

best_loading_matches = (
    cross_method_concordance
    .sort_values(
        "loading_similarity",
        ascending=False,
    )
    .groupby("ica_program", as_index=False)
    .first()
    .rename(
        columns={
            "nmf_program": "best_loading_nmf",
            "ica_pole": "best_loading_pole",
            "loading_similarity": "best_loading_similarity",
            "aligned_score_spearman": "score_at_best_loading",
        }
    )
)

best_score_matches = (
    cross_method_concordance
    .sort_values(
        "aligned_score_spearman",
        ascending=False,
    )
    .groupby("ica_program", as_index=False)
    .first()
    .rename(
        columns={
            "nmf_program": "best_score_nmf",
            "loading_similarity": "loading_at_best_score",
            "aligned_score_spearman": "best_aligned_score_spearman",
        }
    )
)

cross_method_match_summary = (
    best_loading_matches[
        [
            "ica_program",
            "best_loading_nmf",
            "best_loading_pole",
            "best_loading_similarity",
            "score_at_best_loading",
        ]
    ]
    .merge(
        best_score_matches[
            [
                "ica_program",
                "best_score_nmf",
                "loading_at_best_score",
                "best_aligned_score_spearman",
            ]
        ],
        on="ica_program",
    )
)

cross_method_match_summary["same_nmf_across_views"] = (
    cross_method_match_summary["best_loading_nmf"]
    == cross_method_match_summary["best_score_nmf"]
)

cross_method_match_summary.sort_values(
    "best_loading_similarity"
).head(10)

,ica_program,best_loading_nmf,best_loading_pole,best_loading_similarity,score_at_best_loading,best_score_nmf,loading_at_best_score,best_aligned_score_spearman,same_nmf_across_views
30,ICA_PROGRAM_32,NMF_PROGRAM_48,negative,0.544729,0.111432,NMF_PROGRAM_47,0.537310,0.300848,False
1,ICA_PROGRAM_02,NMF_PROGRAM_10,negative,0.565632,0.133306,NMF_PROGRAM_41,0.533539,0.412353,False
3,ICA_PROGRAM_05,NMF_PROGRAM_05,positive,0.577181,-0.007259,NMF_PROGRAM_19,0.572656,0.327340,False
26,ICA_PROGRAM_28,NMF_PROGRAM_31,positive,0.579631,0.348711,NMF_PROGRAM_31,0.579631,0.348711,True
24,ICA_PROGRAM_26,NMF_PROGRAM_29,negative,0.601528,0.308322,NMF_PROGRAM_14,0.505422,0.352715,False
31,ICA_PROGRAM_33,NMF_PROGRAM_05,positive,0.617674,0.080115,NMF_PROGRAM_48,0.497686,0.187190,False
37,ICA_PROGRAM_39,NMF_PROGRAM_49,positive,0.623861,0.276611,NMF_PROGRAM_02,0.461464,0.281025,False
16,ICA_PROGRAM_18,NMF_PROGRAM_31,negative,0.629457,0.207577,NMF_PROGRAM_14,0.514223,0.261552,False
20,ICA_PROGRAM_22,NMF_PROGRAM_48,positive,0.650411,0.133551,NMF_PROGRAM_10,0.489374,0.322222,False
43,ICA_PROGRAM_45,NMF_PROGRAM_07,positive,0.659185,0.175938,NMF_PROGRAM_02,0.585830,0.394452,False


In [33]:
# =============================================================================
# Summarize ICA–NMF cross-method concordance
# =============================================================================

cross_method_overview = pd.Series(
    {
        "recoverable_ica_programs": len(
            recoverable_ica_programs
        ),
        "same_nmf_across_views": (
            cross_method_match_summary[
                "same_nmf_across_views"
            ].sum()
        ),
        "median_best_loading_similarity": (
            cross_method_match_summary[
                "best_loading_similarity"
            ].median()
        ),
        "minimum_best_loading_similarity": (
            cross_method_match_summary[
                "best_loading_similarity"
            ].min()
        ),
        "median_score_at_best_loading": (
            cross_method_match_summary[
                "score_at_best_loading"
            ].median()
        ),
        "minimum_score_at_best_loading": (
            cross_method_match_summary[
                "score_at_best_loading"
            ].min()
        ),
    }
)

cross_method_overview

recoverable_ica_programs           49.000000
same_nmf_across_views              34.000000
median_best_loading_similarity      0.749420
minimum_best_loading_similarity     0.544729
median_score_at_best_loading        0.471874
minimum_score_at_best_loading      -0.007259
dtype: float64

In [34]:
# =============================================================================
# Quantify ICA–NMF match uniqueness
# =============================================================================

match_uniqueness_records = []

for ica_program, program_matches in (
    cross_method_concordance
    .groupby("ica_program")
):
    loading_ranked = program_matches.sort_values(
        "loading_similarity",
        ascending=False,
    )

    score_ranked = program_matches.sort_values(
        "aligned_score_spearman",
        ascending=False,
    )

    match_uniqueness_records.append(
        {
            "ica_program": ica_program,
            "loading_match_margin": (
                loading_ranked.iloc[0]["loading_similarity"]
                - loading_ranked.iloc[1]["loading_similarity"]
            ),
            "score_match_margin": (
                score_ranked.iloc[0]["aligned_score_spearman"]
                - score_ranked.iloc[1]["aligned_score_spearman"]
            ),
        }
    )

match_uniqueness = pd.DataFrame(
    match_uniqueness_records
)

cross_method_match_summary = (
    cross_method_match_summary
    .merge(
        match_uniqueness,
        on="ica_program",
        how="left",
    )
)

cross_method_match_summary[
    [
        "loading_match_margin",
        "score_match_margin",
    ]
].describe()

,loading_match_margin,score_match_margin
count,49.000000,49.000000
mean,0.103596,0.219838
std,0.081645,0.128392
min,0.000157,0.004045
25%,0.042183,0.104451
50%,0.073854,0.240810
75%,0.165905,0.319052
max,0.320388,0.468443


In [35]:
# =============================================================================
# Define convergent ICA–NMF structural matches
# =============================================================================

cross_method_match_summary[
    "cross_method_convergent"
] = (
    cross_method_match_summary[
        "same_nmf_across_views"
    ]
)

cross_method_match_summary[
    [
        "ica_program",
        "best_loading_nmf",
        "best_loading_pole",
        "best_loading_similarity",
        "score_at_best_loading",
        "loading_match_margin",
        "score_match_margin",
        "cross_method_convergent",
    ]
].sort_values(
    [
        "cross_method_convergent",
        "best_loading_similarity",
    ],
    ascending=[
        False,
        False,
    ],
)

,ica_program,best_loading_nmf,best_loading_pole,best_loading_similarity,score_at_best_loading,loading_match_margin,score_match_margin,cross_method_convergent
42,ICA_PROGRAM_44,NMF_PROGRAM_07,positive,0.876205,0.542079,0.226882,0.288843,True
18,ICA_PROGRAM_20,NMF_PROGRAM_45,positive,0.862863,0.583778,0.126127,0.421828,True
40,ICA_PROGRAM_42,NMF_PROGRAM_37,positive,0.849774,0.561692,0.320388,0.327724,True
14,ICA_PROGRAM_16,NMF_PROGRAM_35,positive,0.848743,0.432142,0.242298,0.219131,True
12,ICA_PROGRAM_14,NMF_PROGRAM_40,positive,0.830283,0.496598,0.178745,0.319052,True
46,ICA_PROGRAM_48,NMF_PROGRAM_38,positive,0.828917,0.570089,0.211951,0.339521,True
36,ICA_PROGRAM_38,NMF_PROGRAM_04,positive,0.826121,0.517776,0.140592,0.302844,True
9,ICA_PROGRAM_11,NMF_PROGRAM_28,positive,0.823544,0.508560,0.050272,0.240810,True
27,ICA_PROGRAM_29,NMF_PROGRAM_34,positive,0.806557,0.570501,0.229685,0.389352,True
48,ICA_PROGRAM_50,NMF_PROGRAM_16,positive,0.799162,0.550577,0.070739,0.395918,True


In [36]:
# =============================================================================
# Assemble ICA program discovery summary
# =============================================================================

ica_program_discovery_summary = (
    ica_program_metadata
    .merge(
        cross_method_match_summary,
        left_on="program_id",
        right_on="ica_program",
        how="left",
    )
    .drop(columns="ica_program")
)

ica_program_discovery_summary[
    "program_status"
] = np.where(
    ~ica_program_discovery_summary["is_recoverable"],
    "not_recoverable",
    np.where(
        ica_program_discovery_summary[
            "cross_method_convergent"
        ].fillna(False),
        "candidate_with_cross_method_support",
        "candidate_ica_specific",
    ),
)

ica_program_discovery_summary[
    "program_status"
].value_counts()

program_status
candidate_with_cross_method_support    34
candidate_ica_specific                 15
not_recoverable                         1
Name: count, dtype: int64

In [37]:
# =============================================================================
# Associate recoverable ICA programs with selected phenotype
# =============================================================================

phenotype_associations = []

for program_id in recoverable_ica_programs:
    rho, p_value = spearmanr(
        ica_program_scores[program_id],
        selected_phenotype,
    )

    phenotype_associations.append(
        {
            "program_id": program_id,
            "spearman_rho": rho,
            "p_value": p_value,
        }
    )

phenotype_associations = pd.DataFrame(
    phenotype_associations
)

phenotype_associations[
    "absolute_spearman_rho"
] = phenotype_associations[
    "spearman_rho"
].abs()

phenotype_associations.sort_values(
    "absolute_spearman_rho",
    ascending=False,
).head(10)

,program_id,spearman_rho,p_value,absolute_spearman_rho
35,ICA_PROGRAM_37,0.171233,0.000004,0.171233
31,ICA_PROGRAM_33,0.168724,0.000006,0.168724
13,ICA_PROGRAM_15,-0.164250,0.000011,0.164250
15,ICA_PROGRAM_17,-0.121238,0.001190,0.121238
45,ICA_PROGRAM_47,-0.108656,0.003698,0.108656
34,ICA_PROGRAM_36,-0.108113,0.003874,0.108113
16,ICA_PROGRAM_18,0.106951,0.004277,0.106951
8,ICA_PROGRAM_10,-0.104859,0.005098,0.104859
17,ICA_PROGRAM_19,0.096552,0.009942,0.096552
26,ICA_PROGRAM_28,-0.091576,0.014509,0.091576


In [38]:
# =============================================================================
# Adjust phenotype associations for multiple testing
# =============================================================================

p_values = phenotype_associations[
    "p_value"
].to_numpy()

order = np.argsort(p_values)
ranked_p_values = p_values[order]

adjusted_p_values = (
    ranked_p_values
    * len(ranked_p_values)
    / np.arange(1, len(ranked_p_values) + 1)
)

adjusted_p_values = np.minimum.accumulate(
    adjusted_p_values[::-1]
)[::-1]

q_values = np.empty_like(adjusted_p_values)
q_values[order] = np.clip(
    adjusted_p_values,
    0,
    1,
)

phenotype_associations["q_value"] = q_values

phenotype_associations.sort_values(
    "absolute_spearman_rho",
    ascending=False,
).head(10)

,program_id,spearman_rho,p_value,absolute_spearman_rho,q_value
35,ICA_PROGRAM_37,0.171233,0.000004,0.171233,0.000147
31,ICA_PROGRAM_33,0.168724,0.000006,0.168724,0.000147
13,ICA_PROGRAM_15,-0.164250,0.000011,0.164250,0.000173
15,ICA_PROGRAM_17,-0.121238,0.001190,0.121238,0.014575
45,ICA_PROGRAM_47,-0.108656,0.003698,0.108656,0.029936
34,ICA_PROGRAM_36,-0.108113,0.003874,0.108113,0.029936
16,ICA_PROGRAM_18,0.106951,0.004277,0.106951,0.029936
8,ICA_PROGRAM_10,-0.104859,0.005098,0.104859,0.031222
17,ICA_PROGRAM_19,0.096552,0.009942,0.096552,0.054130
26,ICA_PROGRAM_28,-0.091576,0.014509,0.091576,0.071094


In [39]:
# =============================================================================
# Integrate phenotype associations with program discovery summary
# =============================================================================

program_discovery_summary = (
    ica_program_discovery_summary
    .merge(
        phenotype_associations,
        on="program_id",
        how="left",
    )
)

program_discovery_summary.sort_values(
    "absolute_spearman_rho",
    ascending=False,
).head(10)

,program_id,component_index,median_loading_correlation,minimum_loading_correlation,median_score_correlation,minimum_score_correlation,is_recoverable,best_loading_nmf,best_loading_pole,best_loading_similarity,...,best_aligned_score_spearman,same_nmf_across_views,loading_match_margin,score_match_margin,cross_method_convergent,program_status,spearman_rho,p_value,absolute_spearman_rho,q_value
36,ICA_PROGRAM_37,36,0.886663,0.394542,0.861634,0.303799,True,NMF_PROGRAM_48,positive,0.674587,...,0.378120,False,0.000157,0.063127,False,candidate_ica_specific,0.171233,0.000004,0.171233,0.000147
32,ICA_PROGRAM_33,32,0.952005,0.803028,0.952215,0.814412,True,NMF_PROGRAM_05,positive,0.617674,...,0.187190,False,0.047596,0.005754,False,candidate_ica_specific,0.168724,0.000006,0.168724,0.000147
14,ICA_PROGRAM_15,14,0.999884,0.999691,0.999319,0.998236,True,NMF_PROGRAM_50,positive,0.738967,...,0.469744,True,0.021388,0.175160,True,candidate_with_cross_method_support,-0.164250,0.000011,0.164250,0.000173
16,ICA_PROGRAM_17,16,0.999631,0.998179,0.998953,0.997668,True,NMF_PROGRAM_43,positive,0.737613,...,0.518773,True,0.066256,0.282978,True,candidate_with_cross_method_support,-0.121238,0.001190,0.121238,0.014575
46,ICA_PROGRAM_47,46,0.996821,0.920818,0.986930,0.764861,True,NMF_PROGRAM_12,positive,0.754736,...,0.386766,False,0.105515,0.081182,False,candidate_ica_specific,-0.108656,0.003698,0.108656,0.029936
35,ICA_PROGRAM_36,35,0.904815,0.505311,0.850341,0.591591,True,NMF_PROGRAM_48,negative,0.867431,...,0.526010,False,0.097601,0.095368,False,candidate_ica_specific,-0.108113,0.003874,0.108113,0.029936
17,ICA_PROGRAM_18,17,0.828310,0.514682,0.814200,0.342305,True,NMF_PROGRAM_31,negative,0.629457,...,0.261552,False,0.001938,0.034817,False,candidate_ica_specific,0.106951,0.004277,0.106951,0.029936
9,ICA_PROGRAM_10,9,0.999857,0.999724,0.999510,0.998789,True,NMF_PROGRAM_24,positive,0.738285,...,0.546167,True,0.043395,0.395965,True,candidate_with_cross_method_support,-0.104859,0.005098,0.104859,0.031222
18,ICA_PROGRAM_19,18,0.987469,0.963780,0.972922,0.928770,True,NMF_PROGRAM_33,positive,0.763546,...,0.480296,True,0.103441,0.248485,True,candidate_with_cross_method_support,0.096552,0.009942,0.096552,0.054130
27,ICA_PROGRAM_28,27,0.847782,0.393987,0.862467,0.424554,True,NMF_PROGRAM_31,positive,0.579631,...,0.348711,True,0.057688,0.107277,True,candidate_with_cross_method_support,-0.091576,0.014509,0.091576,0.071094


In [40]:
# =============================================================================
# Identify phenotype-associated candidate programs
# =============================================================================

PHENOTYPE_FDR_THRESHOLD = 0.05

program_discovery_summary[
    "passes_phenotype_fdr"
] = (
    program_discovery_summary["q_value"]
    .lt(PHENOTYPE_FDR_THRESHOLD)
)

program_discovery_summary.loc[
    program_discovery_summary["passes_phenotype_fdr"],
    [
        "program_id",
        "program_status",
        "spearman_rho",
        "q_value",
    ],
].sort_values(
    "spearman_rho",
    key=abs,
    ascending=False,
)

,program_id,program_status,spearman_rho,q_value
36,ICA_PROGRAM_37,candidate_ica_specific,0.171233,0.000147
32,ICA_PROGRAM_33,candidate_ica_specific,0.168724,0.000147
14,ICA_PROGRAM_15,candidate_with_cross_method_support,-0.164250,0.000173
16,ICA_PROGRAM_17,candidate_with_cross_method_support,-0.121238,0.014575
46,ICA_PROGRAM_47,candidate_ica_specific,-0.108656,0.029936
35,ICA_PROGRAM_36,candidate_ica_specific,-0.108113,0.029936
17,ICA_PROGRAM_18,candidate_ica_specific,0.106951,0.029936
9,ICA_PROGRAM_10,candidate_with_cross_method_support,-0.104859,0.031222


In [41]:
# =============================================================================
# Assemble program-discovery metadata
# =============================================================================

program_discovery_metadata = {
    "n_models": len(model_ids),
    "n_input_genes": expression.shape[1],
    "n_discovery_genes": N_DISCOVERY_GENES,
    "n_components": N_COMPONENTS,
    "random_seed": RANDOM_SEED,
    "ica": {
        "n_runs": N_ICA_RUNS,
        "reference_seed": reference_ica_run["seed"],
        "stability_threshold": ICA_STABILITY_THRESHOLD,
        "n_recoverable": int(
            ica_program_metadata["is_recoverable"].sum()
        ),
    },
    "nmf": {
        "n_runs": N_NMF_RUNS,
        "reference_seed": reference_nmf_run["seed"],
        "stability_threshold": NMF_STABILITY_THRESHOLD,
        "n_recoverable": int(
            nmf_program_metadata["is_recoverable"].sum()
        ),
    },
    "cross_method": {
        "n_convergent_ica_programs": int(
            program_discovery_summary[
                "cross_method_convergent"
            ].fillna(False).sum()
        ),
    },
    "phenotype_association": {
        "phenotype": "selected_phenotype",
        "method": "spearman",
        "fdr_method": "BH",
        "fdr_threshold": PHENOTYPE_FDR_THRESHOLD,
        "n_fdr_candidates": int(
            program_discovery_summary[
                "passes_phenotype_fdr"
            ].sum()
        ),
    },
}

In [42]:
# =============================================================================
# Program-discovery output paths
# =============================================================================

ICA_SCORES_OUTPUT_PATH = (
    OUTPUT_DIR / "310_ica_program_scores.parquet"
)

ICA_LOADINGS_OUTPUT_PATH = (
    OUTPUT_DIR / "310_ica_program_loadings.parquet"
)

PHENOTYPE_ASSOCIATIONS_OUTPUT_PATH = (
    OUTPUT_DIR / "310_program_phenotype_associations.csv"
)

METADATA_OUTPUT_PATH = (
    OUTPUT_DIR / "310_program_discovery_metadata.json"
)

In [43]:
# =============================================================================
# Save program-discovery artifacts
# =============================================================================

ica_program_scores.to_parquet(
    ICA_SCORES_OUTPUT_PATH,
    index=False,
)

ica_program_loadings.to_parquet(
    ICA_LOADINGS_OUTPUT_PATH,
    index=False,
)

program_discovery_summary.to_csv(
    PHENOTYPE_ASSOCIATIONS_OUTPUT_PATH,
    index=False,
)

with open(
    METADATA_OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        program_discovery_metadata,
        file,
        indent=2,
    )

print("Program-discovery artifacts written.")
print(f"ICA scores            : {project_relative_path(ICA_SCORES_OUTPUT_PATH)}")
print(f"ICA loadings          : {project_relative_path(ICA_LOADINGS_OUTPUT_PATH)}")
print(f"Phenotype associations: {project_relative_path(PHENOTYPE_ASSOCIATIONS_OUTPUT_PATH)}")
print(f"Metadata              : {project_relative_path(METADATA_OUTPUT_PATH)}")

Program-discovery artifacts written.
ICA scores            : data/processed/cellline_programs/310_ica_program_scores.parquet
ICA loadings          : data/processed/cellline_programs/310_ica_program_loadings.parquet
Phenotype associations: data/processed/cellline_programs/310_program_phenotype_associations.csv
Metadata              : data/processed/cellline_programs/310_program_discovery_metadata.json


## Summary

Using 712 models and 5,000 genes, ICA and NMF yielded 49 recoverable ICA
programs and 48 recoverable NMF factors. Of the 49 recoverable ICA programs,
34 showed structural support from NMF.

Eight ICA programs were associated with the model-level median `LN_IC50`
at `q < 0.05` after Benjamini–Hochberg correction. `ICA_PROGRAM_37` had the
largest absolute Spearman correlation (ρ = 0.171). Notebook 311 evaluates
the robustness of these eight associations.